# Dynamic Breakpoint with LangSmith API

## Steps: 
1. Have Agent.py file ready in the studio folder
2. keep .env file with LangSmith API keys and GOOGle API keys ready
3. set langgraph.json file ready
4. To start the local development server,
   run the following command in your terminal in the /studio directory in this module: BAsh
   ' langgraph dev '

### check for output:
 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs

This in-memory server is designed for development and testing.
For production use, please use LangSmith Deployment.

6. Copy url = http://127.0.0.1:2024

#### Note: 
- check for more info : https://docs.langchain.com/langsmith/quick-start-studio#local-development-server
- check for LangSmith setup guide in the begining of the course.

7. ### Connecting to the Local Server Using the SDK
You use the LangGraph SDK to talk to your local agent: python
```
from langgraph_sdk import get_client

url = "http://127.0.0.1:2024"
client = get_client(url=url)
```
This client object lets you:

    - list assistants
    - create threads
    - run your agent
    - stream results

8. ### Create a thread for storing event checkpoints?
A thread is a conversation session.

It stores:

    - messages
    - memory
    - state
    - checkpoints

You create one like this: python
```
thread = await client.threads.create()
```
This returns: python
```
{"thread_id": "abc123"}
```
You will use this thread ID for the entire conversation.

9.  ### Running the Agent With Dynamic breakpoints
note: learn more here[API breakpoints](https://docs.langchain.com/langsmith/add-human-in-the-loop)

You run your agent like this: python
```
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=initial_input,
    stream_mode="values",
    ):
```
Note:  We can add approval = interrupt("Do you approve?") inside a node while definign the node function. 
- file must be in the Studio folder in .py format .

#### What happens?
Your agent runs step by step:

    - Noide runs and gets interrupted whre th interrupt exists.
    - Run the command with JSON output
    - State is updated with the command
    - Node runs
    - Final answer
After each step, LangGraph sends you a chunk containing the updated state.

In [11]:
from langgraph_sdk import get_client

url = "http://127.0.0.1:2024"
client  = get_client(url=url)

# Search all hosted graphs
assistants = await client.assistants.search()
# list the agents
assistants

[{'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca',
  'graph_id': 'agent',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'agent',
  'created_at': '2026-05-01T21:39:15.638281+00:00',
  'updated_at': '2026-05-01T21:39:15.638281+00:00',
  'version': 1,
  'description': None},
 {'assistant_id': '09f72e2a-1f9e-53ce-9257-2a01f9362e09',
  'graph_id': 'Dynamic_Breakpoints',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'Dynamic_Breakpoints',
  'created_at': '2026-05-01T21:39:15.638281+00:00',
  'updated_at': '2026-05-01T21:39:15.638281+00:00',
  'version': 1,
  'description': None}]

##  Agent - Dynamic_Breakpoint 
- Use interrupt() fuction in the approval node.
- check out how intrrupt works with LangSmith API

In [12]:
# create a thread
thread = await client.threads.create()
print(thread)

{'thread_id': '019de5ac-7ee7-7932-9efc-998aa94a5415', 'created_at': '2026-05-01T22:33:00.390745+00:00', 'updated_at': '2026-05-01T22:33:00.390745+00:00', 'state_updated_at': '2026-05-01T22:33:00.390745+00:00', 'metadata': {}, 'status': 'idle', 'config': {}, 'values': None}


In [13]:
input_dict = {"input": "Tell em a joke!"}

async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="Dynamic_Breakpoints",
    input=input_dict,
    stream_mode="values"):
    
    print(f"Receiving new event of type: {chunk.event}...")
    print(chunk.data)
    print("\n\n")

Receiving new event of type: metadata...
{'run_id': '019de5ac-92ba-7ff3-b0a4-76c98ee6ea21', 'attempt': 1}



Receiving new event of type: values...
{'input': 'Tell em a joke!'}



Receiving new event of type: values...
{'input': 'Tell em a joke!', '__interrupt__': [{'value': 'Do you want to approive this action?', 'id': 'e741e8e3b19633bf7071be93cc3d870a'}]}





## Current state


In [14]:
state = await client.threads.get_state(thread['thread_id'])
state

{'values': {'input': 'Tell em a joke!'},
 'next': ['approval_node'],
 'tasks': [{'id': 'aaa61474-4e76-4f22-538e-ecffa980a66a',
   'name': 'approval_node',
   'path': ['__pregel_pull', 'approval_node'],
   'error': None,
   'interrupts': [{'id': 'e741e8e3b19633bf7071be93cc3d870a',
     'value': 'Do you want to approive this action?'}],
   'checkpoint': None,
   'state': None,
   'result': None}],
 'metadata': {'graph_id': 'Dynamic_Breakpoints',
  'assistant_id': '09f72e2a-1f9e-53ce-9257-2a01f9362e09',
  'user_id': '',
  'created_by': 'system',
  'run_attempt': 1,
  'langgraph_version': '1.1.9',
  'langgraph_api_version': '0.8.1',
  'langgraph_plan': 'enterprise',
  'langgraph_host': 'self-hosted',
  'langgraph_api_url': 'http://127.0.0.1:2024',
  'run_id': '019de5ac-92ba-7ff3-b0a4-76c98ee6ea21',
  'thread_id': '019de5ac-7ee7-7932-9efc-998aa94a5415',
  'source': 'loop',
  'step': 0,
  'parents': {},
  'langgraph_auth_user_id': '',
  'langgraph_request_id': 'ce1d7fce-902d-48f3-9677-fa0494

## Resume the graph run with client.runs.wait(...)
1️⃣ Capture the interrupt ID
From your stream:

python
interrupt_id = chunk.data["__interrupt"][0]["id"]

2️⃣ Send the resume value
Example: user approves → True

#### This tells LangSmith:

- “Continue the run”
- “Inside the node, return True from interrupt()”

In [20]:
#Interrupt_id = state['interrupts'][0]['id']
from langgraph_sdk.schema import Command


await client.runs.wait(
    thread['thread_id'],
    assistant_id = 'Dynamic_Breakpoints',
    command=Command(resume="YES")   
)
    



{'input': 'can eggs crack jokes? NO! they crack each other', 'approved': 'YES'}

In [18]:
state = await client.threads.get_state(thread['thread_id'])
state

{'values': {'input': 'can eggs crack jokes? NO! they crack each other',
  'approved': 'YES'},
 'next': [],
 'tasks': [],
 'metadata': {'graph_id': 'Dynamic_Breakpoints',
  'assistant_id': '09f72e2a-1f9e-53ce-9257-2a01f9362e09',
  'user_id': '',
  'created_by': 'system',
  'run_attempt': 1,
  'langgraph_version': '1.1.9',
  'langgraph_api_version': '0.8.1',
  'langgraph_plan': 'enterprise',
  'langgraph_host': 'self-hosted',
  'langgraph_api_url': 'http://127.0.0.1:2024',
  'run_id': '019de5ad-c961-7273-bcc2-53d601e39c77',
  'thread_id': '019de5ac-7ee7-7932-9efc-998aa94a5415',
  'source': 'loop',
  'step': 2,
  'parents': {},
  'langgraph_auth_user_id': '',
  'langgraph_request_id': 'dda2dd8c-8152-421f-9a7c-d0f3e20e01ea'},
 'created_at': '2026-05-01T22:34:25.647977+00:00',
 'checkpoint': {'checkpoint_id': '1f145ade-86f8-6e1b-8002-572037775de1',
  'thread_id': '019de5ac-7ee7-7932-9efc-998aa94a5415',
  'checkpoint_ns': ''},
 'parent_checkpoint': {'checkpoint_id': '1f145ade-86f6-6056-8001-